In [ ]:
# 03_experiment_analysis

Run quick experiments (train small models), compare LSTM / AttentionLSTM / ConvLSTM, plot results and compute MAE.
This notebook is suitable for quick prototyping; for full experiments use the `src/training/train.py` CLI.


In [ ]:
# Setup and imports
import os, sys, time
project_root = r"C:\Users\shiva\Documents\multicommodity-prices-indonesia"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from pathlib import Path
plt.rcParams['figure.figsize'] = (9,5)

# Import modules
from src.models.lstm_model import SimpleLSTM
from src.models.attention_lstm import AttentionLSTM
from src.models.conv_lstm import ConvLSTM
from src.training.dataset import SeqDataset
from src.training.evaluate import mae

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
proc = Path(project_root)/"data"/"processed"
X_train = np.load(proc/"X_train.npy")
y_train = np.load(proc/"y_train.npy")
X_test = np.load(proc/"X_test.npy")
y_test = np.load(proc/"y_test.npy")
print("Shapes:", X_train.shape, y_train.shape, X_test.shape, y_test.shape)
n_features = X_train.shape[2]


In [ ]:
import torch.nn as nn
import torch.optim as optim

def quick_train(model, X_tr, y_tr, X_val, y_val, epochs=10, batch_size=32, lr=1e-3):
    model = model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.L1Loss()
    train_ds = SeqDataset(X_tr, y_tr)
    val_ds = SeqDataset(X_val, y_val)
    tr_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    history = {"train_loss": [], "val_mae": []}
    for ep in range(1, epochs+1):
        model.train()
        running = 0.0
        n = 0
        for xb, yb in tr_dl:
            xb = xb.to(device); yb = yb.to(device)
            opt.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            opt.step()
            running += loss.item()*xb.size(0)
            n += xb.size(0)
        train_loss = running / n
        # eval
        model.eval()
        preds = []
        trues = []
        with torch.no_grad():
            for xb, yb in val_dl:
                xb = xb.to(device)
                out = model(xb)
                preds.append(out.cpu().numpy()); trues.append(yb.numpy())
        preds = np.vstack(preds); trues = np.vstack(trues)
        val_mae = mae(trues, preds)
        history["train_loss"].append(train_loss)
        history["val_mae"].append(val_mae)
        print(f"Ep {ep}/{epochs} train_loss={train_loss:.6f} val_mae={val_mae:.6f}")
    return model, history


In [ ]:
# limit dataset for quick runs (optional)
# To run fast during prototyping, you can subsample training set:
subsample = None  # set to int like 200 for speed; or None for full data
if subsample is not None:
    X_tr = X_train[:subsample]; y_tr = y_train[:subsample]
else:
    X_tr, y_tr = X_train, y_train
X_val, y_val = X_test, y_test

# Model 1: Simple LSTM
model_lstm = SimpleLSTM(input_size=n_features, hidden_size=64, num_layers=1, out_size=n_features, batchnorm=True)
print("Training SimpleLSTM...")
m1, h1 = quick_train(model_lstm, X_tr, y_tr, X_val, y_val, epochs=8, batch_size=64, lr=1e-3)

# Model 2: Attention LSTM
model_att = AttentionLSTM(input_size=n_features, hidden_size=64, num_layers=1, out_size=n_features, batchnorm=True)
print("Training AttentionLSTM...")
m2, h2 = quick_train(model_att, X_tr, y_tr, X_val, y_val, epochs=8, batch_size=64, lr=1e-3)

# Model 3: ConvLSTM
model_conv = ConvLSTM(input_size=n_features, conv_channels=[32,64], kernel_size=3, lstm_hidden=64, lstm_layers=1, out_size=n_features, batchnorm=True)
print("Training ConvLSTM...")
m3, h3 = quick_train(model_conv, X_tr, y_tr, X_val, y_val, epochs=8, batch_size=64, lr=1e-3)


In [ ]:
plt.figure(figsize=(10,4))
plt.plot(h1['val_mae'], label='LSTM val_mae')
plt.plot(h2['val_mae'], label='Attention val_mae')
plt.plot(h3['val_mae'], label='ConvLSTM val_mae')
plt.xlabel("Epoch")
plt.ylabel("Validation MAE (normalized)")
plt.legend()
plt.title("Validation MAE (quick runs)")
plt.show()


In [ ]:
from src.data import preprocess
# evaluate on test (full)
def predict_and_eval(model, X, y):
    model.eval()
    dl = DataLoader(SeqDataset(X,y), batch_size=128, shuffle=False)
    preds = []
    trues = []
    with torch.no_grad():
        for xb, yb in dl:
            xb = xb.to(device)
            out = model(xb).cpu().numpy()
            preds.append(out); trues.append(yb.numpy())
    preds = np.vstack(preds); trues = np.vstack(trues)
    return trues, preds

for name, m in [('LSTM', m1), ('AttLSTM', m2), ('ConvLSTM', m3)]:
    trues, preds = predict_and_eval(m, X_test, y_test)
    val_mae = mae(trues, preds)
    # denormalize
    scaler_path = Path(project_root)/"data"/"processed"/"scaler.json"
    import json
    with open(scaler_path,"r") as f:
        scaler = json.load(f)
    cols = list(pd.read_csv(Path(project_root)/"data"/"csv"/"pihps_clean.csv", index_col=0).columns)
    trues_den = preprocess.denormalize(trues, cols, scaler)
    preds_den = preprocess.denormalize(preds, cols, scaler)
    print(f"{name} normalized MAE: {val_mae:.6f}")
    print(f"{name} denormalized MAE (overall): {mae(trues_den, preds_den):.4f}")
    # per-feature
    pf = np.mean(np.abs(trues_den - preds_den), axis=0)
    # show top 5 worst features
    worst_idx = np.argsort(-pf)[:5]
    print("Top 5 worst features (MAE):")
    for idx in worst_idx:
        print(f"  {cols[idx]}: {pf[idx]:.4f}")
    print("-"*40)


In [ ]:
# plot predictions vs true for first 4 features (denormalized)
num_plot = min(4, len(cols))
t = np.arange(trues_den.shape[0])
fig, axes = plt.subplots(num_plot, 1, figsize=(10, 3*num_plot), sharex=True)
for i in range(num_plot):
    axes[i].plot(t, trues_den[:, i], label='true')
    axes[i].plot(t, preds_den[:, i], label='pred')
    axes[i].set_title(cols[i])
    axes[i].legend()
plt.tight_layout()


In [ ]:
## Notes / next steps

- Use `src/training/train.py` for production experiments (longer epochs, checkpoints, LR schedules, early stopping).
- For reproducibility set seeds (`torch.manual_seed(23)`, `np.random.seed(23)`).
- Expand experiments: hyperparameter grid, feature selection, longer sequence lengths, multi-step forecasting.
